# 01_clean_tsac

TSAC cleaning workflow (explicit, reproducible, and aligned with team shared utilities).

In [ ]:
# Works both locally (repo root or notebooks/) and in Colab after cloning the repo.
import os, sys
from pathlib import Path
if Path('notebooks').is_dir():
    ROOT = Path('.').resolve()
elif Path('../notebooks').is_dir():
    ROOT = Path('..').resolve()
else:  # Colab: clone first, e.g. !git clone https://github.com/<owner>/tunisian-sentiment.git
    ROOT = Path('/content/tunisian-sentiment')
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Repo root:', ROOT)


In [ ]:
import pandas as pd
from src.load_raw import load_tsac
from src.cleaning_utils import fix_text, has_letter, CleaningLog

OUT = ROOT / 'data/interim/tsac_clean.csv'
LOG = ROOT / 'logs/cleaning_log_tsac.csv'
log = CleaningLog()


In [ ]:
train = load_tsac('train', ROOT)
test = load_tsac('test', ROOT)
log.add('load TSAC train', len(train), len(train), 'shared loader load_tsac(train)')
log.add('load TSAC test', len(test), len(test), 'shared loader load_tsac(test)')
print({'train_raw': len(train), 'test_raw': len(test)})
assert len(train) == 13669 and len(test) == 3400, 'unexpected raw TSAC counts'

df = pd.concat([train, test], ignore_index=True)
print('Split counts after concat:', df.split.value_counts().to_dict())
df.head()


In [ ]:
print('Raw label values:', sorted(df['label'].dropna().unique()))
print('Raw label counts:', df['label'].value_counts(dropna=False).to_dict())

LABEL_MAP = {'pos': 'pos', 'neg': 'neg'}
before = len(df)
df['label'] = df['label'].map(LABEL_MAP)
unknown = int(df['label'].isna().sum())
assert unknown == 0, f'Unexpected labels after mapping: {unknown}'
log.add('normalize labels to pos/neg', before, len(df), 'strict map {pos->pos, neg->neg}')


In [ ]:
n_x000d = df['text'].str.contains('_x000D_', regex=False).sum()
changed = (df['text'] != df['text'].map(fix_text)).sum()
df['text'] = df['text'].map(fix_text)
log.add('fix_text (_x000D_, spaces)', len(df), len(df), f'{n_x000d} rows had _x000D_; {changed} rows changed')


In [ ]:
before = len(df)
df = df[df['text'] != ''].copy()
log.add('remove empty rows', before, len(df))

before = len(df)
df = df[df['text'].map(has_letter)].copy()
log.add('remove rows without any letter', before, len(df))


In [ ]:
conflict_texts = df.groupby('text')['label'].nunique()
conflict_texts = conflict_texts[conflict_texts > 1].index
conflict_count = len(conflict_texts)
before = len(df)
df = df[~df['text'].isin(conflict_texts)].copy()
removed_conflict_rows = before - len(df)
log.add('remove label conflicts (all copies)', before, len(df), f'{conflict_count} conflict texts')
print({'conflict_texts': conflict_count, 'rows_removed': removed_conflict_rows})


In [ ]:
duplicate_text_rows_anywhere = int(df.duplicated(subset='text', keep='first').sum())
print({'duplicate_text_rows_anywhere': duplicate_text_rows_anywhere, 'proposal_reference': 1791, 'difference_vs_reference': duplicate_text_rows_anywhere - 1791})
if duplicate_text_rows_anywhere != 1791:
    print('Note: duplicate-text result differs from the proposal reference; reported as observed without forcing a match.')

before = len(df)
df = df.drop_duplicates(subset=['split', 'text'], keep='first').copy()
log.add('deduplicate within split (split,text)', before, len(df))


In [ ]:
train_texts = set(df.loc[df['split'] == 'train', 'text'])
test_texts = set(df.loc[df['split'] == 'test', 'text'])
leak_texts = train_texts & test_texts
leak_count = len(leak_texts)
before = len(df)
df = df[~((df['split'] == 'train') & (df['text'].isin(leak_texts)))].copy()
log.add('drop train copies that leak into test', before, len(df), f'{leak_count} leaked texts; kept official test copies')
print({'leaked_texts_between_splits': leak_count, 'train_rows_removed_for_leakage': before - len(df)})


In [ ]:
df['source'] = 'tsac'
df = df[['id', 'text', 'label', 'source', 'split']]

assert (df['text'] != '').all(), 'empty text rows remain'
assert set(df['label'].unique()) <= {'pos', 'neg'}, 'labels must be pos/neg only'
assert df['id'].is_unique, 'ids must stay unique'
assert not df['text'].duplicated().any(), 'duplicate text rows remain'
assert not set(df.loc[df['split'] == 'train', 'text']) & set(df.loc[df['split'] == 'test', 'text']), 'train/test leakage remains'

OUT.parent.mkdir(parents=True, exist_ok=True)
LOG.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False, encoding='utf-8')
log.add('final tsac_clean.csv', len(df), len(df))
log.save(LOG)

print('Final counts by split:', df['split'].value_counts().to_dict())
print('Final counts by label:', df['label'].value_counts().to_dict())
print('Saved:', OUT)
print('Saved:', LOG)
log.to_frame()
